In [1]:
# Setup PyCUDA (Colab / Linux)
!pip install pycuda

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 26.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 11.1 MB/s eta 0:00:00
  Created wheel for pycuda: filename=pycuda-2026.1-cp312-cp312-linux_x86_64.whl size=659447 sha256=4e0f6f9af0977f23b66bd1292829a68c09e49ff19583432419548c0ad1192341
  Stored in directory: /root/.cache/pip/wheels/90/2a/71/75ec0cc316cc0ff494bfffa2935e02580129cb7f859a0cfd8f
Successfully built pycuda


In [2]:
#Cek GPU
!nvidia-smi

Thu Jun 11 00:49:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# **CUDA Matrix Multiplication (PyCUDA)**

Matrix multiplication is one of the most important workloads in GPU computing because it demonstrates:

* Massive parallelism
* Grid + block 2D indexing
* Memory optimization opportunities (shared memory later)


In [3]:
# CPU Matrix Multiplication (Baseline)
import numpy as np
import time

def cpu_matmul(A, B):
    start = time.time()
    C = np.dot(A, B)
    end = time.time()
    return C, end - start

In [4]:
# CUDA Kernel (Grid + Block 2D Indexing)

import pycuda.autoinit
import pycuda.driver as cuda
import numpy as np
from pycuda.compiler import SourceModule
import time

mod = SourceModule("""
__global__ void matmul(float *A, float *B, float *C, int N)
{
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < N && col < N)
    {
        float value = 0;

        for (int k = 0; k < N; k++)
        {
            value += A[row * N + k] * B[k * N + col];
        }

        C[row * N + col] = value;
    }
}
""")

func = mod.get_function("matmul")

In [5]:
# GPU Matrix Multiplication Function
def gpu_matmul(A, B):
    N = A.shape[0]
    C = np.zeros((N, N), dtype=np.float32)

    A_gpu = cuda.mem_alloc(A.nbytes)
    B_gpu = cuda.mem_alloc(B.nbytes)
    C_gpu = cuda.mem_alloc(C.nbytes)

    cuda.memcpy_htod(A_gpu, A)
    cuda.memcpy_htod(B_gpu, B)

    block = (16, 16, 1)
    grid = ((N + 15)//16, (N + 15)//16)

    start = cuda.Event()
    end = cuda.Event()

    start.record()

    func(A_gpu, B_gpu, C_gpu, np.int32(N),
         block=block,
         grid=grid)

    end.record()
    end.synchronize()

    cuda.memcpy_dtoh(C, C_gpu)

    gpu_time = start.time_till(end) / 1000
    return C, gpu_time

In [6]:
# Benchmark CPU vs GPU (We test different matrix sizes)
sizes = [64, 128, 256, 512]

cpu_times = []
gpu_times = []

for N in sizes:
    A = np.random.rand(N, N).astype(np.float32)
    B = np.random.rand(N, N).astype(np.float32)

    _, t_cpu = cpu_matmul(A, B)
    _, t_gpu = gpu_matmul(A, B)

    cpu_times.append(t_cpu)
    gpu_times.append(t_gpu)

    print(f"N={N} | CPU={t_cpu:.4f}s | GPU={t_gpu:.4f}s")

N=64 | CPU=0.0018s | GPU=0.0004s
N=128 | CPU=0.0028s | GPU=0.0001s
N=256 | CPU=0.0005s | GPU=0.0002s
N=512 | CPU=0.0026s | GPU=0.0012s
